In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import numpy as np
import pandas as pd 
import cv2 
import io
import random
from torch.utils.data import Dataset, DataLoader
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Dense, Reshape, BatchNormalization, Input, Conv2D, MaxPool2D, Lambda, Bidirectional, Dropout, LSTM
# from tensorflow.compat.v1.keras.layers import CuDNNLSTM
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import backend as K
import matplotlib.pyplot as plt
from itertools import groupby

In [2]:
# parameter
alphabets = "0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ "
max_str_len = 19  # max length of input labels
num_of_characters = len(alphabets) + 1  # +1 for ctc pseudo blank
num_of_timestamps = 64  # max length of predicted labels
default_path = r"../Datasets/iam_words/"
# batch size
batch_size = 512

def label_to_num(txt):
    # encoding each output word into digits
    dig_lst = []
    
    for index, char in enumerate(txt):
        try:
            dig_lst.append(alphabets.index(char))
        except:
            print(char)
    
    return pad_sequences([dig_lst], maxlen=max_str_len, padding='post', value=len(alphabets))[0]

def ctc_decoder(predictions):
    '''
    input: given batch of predictions from text rec model
    output: return lists of raw extracted text

    '''
    text_list = []
    
    pred_indcies = np.argmax(predictions, axis=2)
    
    for i in range(pred_indcies.shape[0]):
        ans = ""
        
        ## merge repeats
        merged_list = [k for k,_ in groupby(pred_indcies[i])]
        
        ## remove blanks
        for p in merged_list:
            if p != len(alphabets):
                ans += alphabets[int(p)]
        
        text_list.append(ans)
        
    return text_list

def num_to_label(num):
    ret = ""
    for ch in num:
        if ch == -1:  # CTC Blank
            break
        else:
            ret += alphabets[ch]
    return ret
print(batch_size)

def process_single_sample(img_path, label):

    # 1. Read image
    img = tf.io.read_file(img_path)

    # 2. Decode and convert to grayscale
    img = tf.io.decode_png(img, channels=1)

    # 3. Convert to float32 in [0, 1] range
    img = tf.image.convert_image_dtype(img, tf.float32)

    # 4. Resize to the desired size
    img = tf.image.resize(img, [32, 128])
    
#     img = tf.transpose(img, perm=[1, 0, 2])
    return {"image": img, "label": label}

512


In [3]:
import pandas as pd
import os

dataset = []

with open(r"../Datasets/iam_words/words.txt", "r") as f:

    for line in f:

        if line.startswith("#"):
            continue

        parts = line.strip().split()

        if len(parts) < 9:
            continue

        word_id = parts[0]
        status = parts[1]

        if status != "ok":
            continue

        label = " ".join(parts[8:])

        folder1 = word_id.split("-")[0]
        folder2 = "-".join(word_id.split("-")[:2])

        image_path = os.path.join(
            "../Datasets/iam_words",
            "words",
            folder1,
            folder2,
            f"{word_id}.png"
        )

        if os.path.exists(image_path):
            dataset.append([image_path, label])

data = pd.DataFrame(
    dataset,
    columns=["Fpath", "Identify"]
).astype(str)

data.dropna(axis=0, inplace=True)

print(data.shape)

train = data.sample(
    frac=0.9,
    random_state=42
)

unique_train = train["Fpath"].unique()

valid = data.drop(train.index)

print(train.shape)
print(valid.shape)
print(data.shape)

(38305, 2)
(34474, 2)
(3831, 2)
(38305, 2)


In [4]:
import os

print(os.path.exists('../Datasets/iam_words/data.xlsx'))

False


In [5]:
train = train[0:80000]
valid = valid[0:8000]
# reset index
train.reset_index(inplace=True, drop=True)
valid.reset_index(inplace=True, drop=True)

vocab = set("".join(map(str, valid['Identify'])))
print(sorted(vocab))
vocab = set("".join(map(str, train['Identify'])))
print(sorted(vocab))

[' ', '!', '"', '#', "'", '(', ')', '*', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'Y', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
[' ', '!', '"', '#', "'", '(', ')', '*', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [6]:
# size of dataset
train_size = 80000
valid_size = 8000
print(valid_size)

8000


In [7]:
train_x = []
valid_x = []

for i in range(len(valid)):
    valid_x.append(
        valid.iloc[i]["Fpath"]
    )

for i in range(len(train)):
    train_x.append(
        train.iloc[i]["Fpath"]
    )

valid_y = []

for i in range(len(valid)):
    valid_y.append(
        label_to_num(
            valid.iloc[i]["Identify"]
        )
    )

train_y = []

for i in range(len(train)):
    train_y.append(
        label_to_num(
            train.iloc[i]["Identify"]
        )
    )

print(len(valid_y))
print(len(train_y))

.
,
'
.
.
,
(
.
.
.
'
"
.
.
,
.
'
'
-
,
.
.
.
.
.
.
.
.
-
.
-
"
-
.
#
'
'
-
-
.
,
.
.
-
-
-
.
-
,
-
'
.
,
.
.
,
.
.
.
,
,
.
.
-
-
"
.
.
.
.
'
,
.
,
,
.
.
,
,
-
)
"
-
.
.
.
.
.
.
"
'
.
"
.
.
.
'
'
.
.
-
-
,
-
,
.
,
.
,
,
-
-
"
?
.
,
,
"
.
.
,
,
'
,
.
.
.
.
.
"
.
.
.
,
,
,
,
,
.
.
-
,
.
,
,
-
,
.
,
-
#
,
.
.
,
.
.
"
"
,
-
,
.
,
,
.
"
.
,
.
.
.
.
.
'
-
"
.
-
"
.
.
.
"
.
,
.
,
,
(
)
.
.
.
"
'
,
.
-
.
'
?
.
.
.
.
.
.
.
.
.
-
,
.
.
.
-
.
,
.
:
-
.
-
;
.
,
,
"
.
.
-
.
.
.
.
-
.
.
.
,
.
.
.
,
"
-
.
.
.
.
.
!
.
.
,
-
.
'
,
-
-
:
,
.
!
.
"
-
!
!
,
#
.
"
,
,
-
,
,
-
-
"
"
,
"
'
.
,
"
.
,
-
,
,
,
,
-
-
.
.
-
.
.
.
-
-
.
.
.
.
,
"
-
,
"
.
.
,
-
)
,
(
,
.
.
.
'
.
-
.
,
,
.
-
,
.
"
.
.
.
-
-
-
,
,
,
.
.
"
.
-
.
;
,
,
,
.
.
,
,
,
.
,
-
.
"
.
.
,
.
.
-
.
,
,
-
.
.
;
,
,
,
.
.
,
.
,
.
(
:
.
.
,
.
.
-
.
,
-
.
(
.
,
,
-
-
,
"
.
,
"
.
(
)
"
.
.
"
,
'
.
,
.
,
.
.
.
.
.
,
,
?
-
,
,
*
?
:
'
'
"
.
.
,
,
.
,
,
.
-
.
,
'
,
.
.
.
.
,
.
,
,
-
.
-
,
?
.
.
,
,
.
"
.
.
-
.
:
:
.
,
.
.
,
-
,
,
,
,
,
,
-
.
.
'
,
-
,
-


In [8]:
# ===========================================================================
# DATASET LOADER
# ===========================================================================

def load_iam_samples_final(data_root):

    words_txt = os.path.join(
        data_root,
        'words.txt'
    )

    if not os.path.exists(words_txt):
        print(f"words.txt not found: {words_txt}")
        return []

    samples = []

    with open(words_txt, 'r', encoding='utf-8') as f:

        for line in f:

            if line.startswith('#'):
                continue

            parts = line.strip().split()

            if len(parts) < 9:
                continue

            word_id = parts[0]
            status = parts[1]

            if status != 'ok':
                continue

            label = " ".join(parts[8:])

            folder1 = word_id.split("-")[0]
            folder2 = "-".join(word_id.split("-")[:2])

            img_path = os.path.join(
                data_root,
                'words',
                folder1,
                folder2,
                f'{word_id}.png'
            )

            if os.path.exists(img_path):
                samples.append(
                    (img_path, label)
                )

    print(f"Loaded {len(samples)} samples")

    return samples

In [9]:
train_dataset = tf.data.Dataset.from_tensor_slices((train_x, train_y))

train_dataset = (
    train_dataset.map(
        process_single_sample, num_parallel_calls=tf.data.experimental.AUTOTUNE
    )
    .batch(batch_size)
    .prefetch(buffer_size=tf.data.experimental.AUTOTUNE)
)

valid_dataset = tf.data.Dataset.from_tensor_slices((valid_x, valid_y))
valid_dataset = (
    valid_dataset.map(
        process_single_sample, num_parallel_calls=tf.data.experimental.AUTOTUNE
    )
    .batch(batch_size)
    .prefetch(buffer_size=tf.data.experimental.AUTOTUNE)
)

In [10]:
valid_y[0]

array([22, 24, 27, 14, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63,
       63, 63])

In [11]:
class CTCLayer(layers.Layer):

    def __init__(self, name=None):

        super().__init__(name=name)
        self.loss_fn = K.ctc_batch_cost

    def call(self, y_true, y_pred):
        # Compute the training-time loss value and add it
        # to the layer using `self.add_loss()`.

        batch_len = tf.cast(tf.shape(y_true)[0], dtype="int64")
        input_length = tf.cast(tf.shape(y_pred)[1], dtype="int64")
        label_length = tf.cast(tf.shape(y_true)[1], dtype="int64")

        input_length = input_length * tf.ones(shape=(batch_len, 1), dtype="int64")
        label_length = label_length * tf.ones(shape=(batch_len, 1), dtype="int64")

        loss = self.loss_fn(y_true, y_pred, input_length, label_length)
        self.add_loss(loss)

        # At test time, just return the computed predictions
        return y_pred

In [12]:
input_data = Input(shape=(32, 128, 1), name='image')
labels = layers.Input(name="label", shape=(None,), dtype="float32")

inner = Conv2D(32, (3, 3), padding='same', name='conv1', activation='selu')(input_data)
inner = MaxPool2D(pool_size=(2, 2), name='max1')(inner)

inner = Conv2D(64, (3, 3), padding='same', name='conv2', activation='selu')(inner)
inner = MaxPool2D(pool_size=(2, 2), name='max2')(inner)

inner = Conv2D(128, (3, 3), padding='same', name='conv3', activation='selu')(inner)
inner = Conv2D(128, (3, 3), padding='same', name='conv4', activation='selu')(inner)

inner = Conv2D(512, (3, 3), padding='same', name='conv5', activation='selu')(inner)
inner = Conv2D(512, (3, 3), padding='same', name='conv6', activation='selu')(inner)
inner = Dropout(0.2)(inner)

inner = Conv2D(512, (3, 3), padding='same', name='conv7', activation='selu')(inner)
inner = Conv2D(512, (3, 3), padding='same', name='conv8', activation='selu')(inner)
inner = MaxPool2D(pool_size=(2, 1), name='max8')(inner)

inner = Conv2D(256, (3, 3), padding='same', name='conv9',  activation='selu')(inner)
inner = BatchNormalization()(inner)
inner = Dropout(0.2)(inner)

inner = Conv2D(256, (3, 3), padding='same', name='conv10', activation='selu')(inner)
inner = BatchNormalization()(inner)
inner = MaxPool2D(pool_size=(2, 1), name='max10')(inner)
inner = Dropout(0.2)(inner)

inner = Conv2D(64, (2,2), name='conv11', activation='selu')(inner)
inner = Dropout(0.2)(inner)

# CNN to RNN
squeezed = Lambda(lambda x: K.squeeze(x, 1))(inner)
# RNN
inner = Bidirectional(LSTM(128, return_sequences=True), name='lstm1')(squeezed)
inner = Bidirectional(LSTM(512, return_sequences=True), name='lstm2')(inner)
inner = Bidirectional(LSTM(512, return_sequences=True), name='lstm3')(inner)
inner = Bidirectional(LSTM(512, return_sequences=True), name='lstm4')(inner)
inner = Bidirectional(LSTM(128, return_sequences=True), name='lstm5')(inner)
dense_= Dense(128,activation = 'relu')(inner)
# OUTPUT
y_pred = Dense(num_of_characters,activation = 'softmax', name='dense2')(dense_)
output = CTCLayer(name="ctc_loss",)(labels, y_pred)

In [13]:
model = Model(inputs=input_data, outputs=y_pred)
model.summary()

# model for train
train_model = Model(inputs=[input_data, labels], outputs=output)
train_model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 image (InputLayer)          [(None, 32, 128, 1)]      0         
                                                                 
 conv1 (Conv2D)              (None, 32, 128, 32)       320       
                                                                 
 max1 (MaxPooling2D)         (None, 16, 64, 32)        0         
                                                                 
 conv2 (Conv2D)              (None, 16, 64, 64)        18496     
                                                                 
 max2 (MaxPooling2D)         (None, 8, 32, 64)         0         
                                                                 
 conv3 (Conv2D)              (None, 8, 32, 128)        73856     
                                                                 
 conv4 (Conv2D)              (None, 8, 32, 128)        147584

In [2]:
import torch

print("Torch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("CUDA Device Count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch Version: 2.12.1+cu126
CUDA Available: True
CUDA Device Count: 1
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [1]:
import tensorflow as tf

print("TensorFlow Version:", tf.__version__)
print("Built with CUDA:", tf.test.is_built_with_cuda())
print("GPUs:", tf.config.list_physical_devices('GPU'))

TensorFlow Version: 2.10.0
Built with CUDA: True
GPUs: []


In [ ]:
train_model.compile(
    optimizer=Adam(
        learning_rate=0.001,
        beta_1=0.9,
        beta_2=0.999,
        clipnorm=1.0
    ),
    metrics=[tf.keras.metrics.Accuracy()]
)

filepath = "best_model.h5"

checkpoint = ModelCheckpoint(
    filepath=filepath,
    monitor='val_loss',
    verbose=1,
    save_best_only=True,
    save_weights_only=True,
    mode='auto'
)

earlyStopping = EarlyStopping(
    monitor='val_loss',
    mode='auto',
    patience=15
)

callbacks_list = [
    checkpoint,
    earlyStopping
]

history = train_model.fit(
    train_dataset,
    epochs=150,
    validation_data=valid_dataset,
    verbose=1,
    shuffle=True,
    callbacks=callbacks_list
)

model.save('my_model.h5')

14/68 [=====>........................] - ETA: 36:03 - loss: 36.3540 - accuracy: 0.0000e+00